# LLM Basics
In Kaggle: **Settings → Accelerator → GPU T4** and **Internet → On**

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = "Qwen/Qwen2.5-0.5B-Instruct"  # small open LLM (0.5 billion parameters)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
print("Loaded on", device)

# 1. Tokens - how an LLM sees text
Text is split into pieces (tokens) and each piece becomes a number.

In [ ]:
text = "Namibia is a beautiful country."
ids = tokenizer(text)["input_ids"]

print("Token IDs:", ids)
print("Tokens   :", [tokenizer.decode([i]) for i in ids])

# 2. Next word prediction - the core job of an LLM
The model gives a probability for every possible next token.

In [ ]:
text = "The capital of Namibia is the city of"
inputs = tokenizer(text, return_tensors="pt").to(device)

with torch.no_grad():
    logits = model(**inputs).logits[0, -1]  # scores for the next token

probs = logits.softmax(dim=-1)
top = probs.topk(5)
for p, i in zip(top.values, top.indices):
    print(f"{tokenizer.decode([i])!r:15} {p:.1%}")

# 3. Chat with the model
Generating an answer = predicting the next token again and again.

In [ ]:
def ask(prompt, temperature=0.7, max_new_tokens=150):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature)
    print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

In [ ]:
ask("What is Linux? Answer in one sentence.")

# 4. Temperature - low = safe and repetitive, high = creative and random

In [ ]:
for t in [0.1, 1.5]:
    print(f"--- temperature {t} ---")
    for _ in range(3):
        ask("Give me a name for a pet cat. Only the name.", temperature=t, max_new_tokens=10)

# 5. Prompting - a better prompt gives a better answer

In [ ]:
ask("Tell me about Python.")

In [ ]:
ask("You are a teacher. Explain Python to a high-school student in 3 bullet points, max 15 words each.")

# 6. Limits - LLMs can be confidently wrong (hallucination)
The book below is made up, but the model still answers.

In [ ]:
ask("Who wrote the famous novel 'The Silent River of Okahandja'? Answer in two sentences.")  # this book does not exist